In [1]:
import sys
import pandas as pd

print("Python:", sys.version)

# If this fails, you likely need: `pip install -e .` from the repo root
import miaproc
from miaproc.biomass import BiomassColumns, load_packaged_equations, estimate_trees

print("miaproc imported from:", miaproc.__file__)
print("miaproc.biomass is available ✅")

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
miaproc imported from: C:\Users\Julian\work\manglaria\repos\miaproc\src\miaproc\__init__.py
miaproc.biomass is available ✅


In [2]:
CSV_PATH = "../tests/data/biomass/20260303_biomass_test.csv"  

# Mexican state to filter equations (edit this)
ESTADO = "Campeche"

# Expected response variable in the equations parquet
RESPONSE_VARIABLE = "VRTAcc_m3"

# Column mapping (edit if your CSV uses different names)
COLS = BiomassColumns(
    species="Species",
    dbh_cm="DBH (cm)",
    height_m="Total Height (m)",
)

# Range handling: "warn" (default), "clip", "error", or "ignore"
RANGE_POLICY = "warn"

print("CSV_PATH =", CSV_PATH)
print("ESTADO   =", ESTADO)
print("Columns  =", COLS)

CSV_PATH = ../tests/data/biomass/20260303_biomass_test.csv
ESTADO   = Campeche
Columns  = BiomassColumns(species='Species', dbh_cm='DBH (cm)', height_m='Total Height (m)')


In [8]:
# Read input data (csv with tree measurements).

df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

df = df[:100]

df.head(10)

Rows: 2167
Columns: ['study_id', 'site_description', 'site_condition', 'Country', 'Protected area', 'Location', 'Site', 'ManglarIA Project Site ID', 'IDSitio', 'Plot ID', 'plot_area_ha', 'Date of collection', 'Year', 'Month', 'Latitude', 'Longitude', 'position_method', 'Species', 'Species_code ', 'Girth', 'DBH (cm)', 'Juvenil / Adult', 'Total Height (m)', 'Stump Diameter (cm)', 'condition_live_or_dead', 'tree_height_method']


,study_id,site_description,site_condition,Country,Protected area,Location,Site,ManglarIA Project Site ID,IDSitio,Plot ID,...,position_method,Species,Species_code,Girth,DBH (cm),Juvenil / Adult,Total Height (m),Stump Diameter (cm),condition_live_or_dead,tree_height_method
0,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,20.0,dead,NaN
1,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,17.0,dead,NaN
2,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,53.0,dead,NaN
3,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,DBH < 2.5 cm,NaN,Juvenil,1.2,NaN,Live,rod
4,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,62.0,dead,NaN
5,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,44.0,dead,NaN
6,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,20.0,dead,NaN
7,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,NaN,NaN,NaN,NaN,21.0,dead,NaN
8,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,3.5,1.114085,Adult,1.8,NaN,Live,rod
9,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,handheld,Avicennia germinans,Ag,DBH < 2.5 cm,NaN,Juvenil,1.2,NaN,Live,rod


In [9]:
# Load allometric equations from our database (in this case the parquet file).

equations = load_packaged_equations()

print("Equations rows:", len(equations))
print("Equations columns:", list(equations.columns))

equations.head(5)

Equations rows: 20123
Equations columns: ['estado', 'clave_umafor', 'cveecon4', 'nombrecientifico_apg_raw', 'nivel_asignacion', 'clave_ecuacion', 'ecuacion_raw', 'fuente_clave', 'dbh_range_cm_raw', 'height_range_m_raw', 'response_variable', 'ecuacion_normalizada', 'ecuacion_numpy', 'dbh_min_cm', 'dbh_max_cm', 'alt_min_m', 'alt_max_m', 'parse_status', 'parse_notes', 'nivel_asignacion_desc', 'fuente_referencia']


,estado,clave_umafor,cveecon4,nombrecientifico_apg_raw,nivel_asignacion,clave_ecuacion,ecuacion_raw,fuente_clave,dbh_range_cm_raw,height_range_m_raw,...,ecuacion_normalizada,ecuacion_numpy,dbh_min_cm,dbh_max_cm,alt_min_m,alt_max_m,parse_status,parse_notes,nivel_asignacion_desc,fuente_referencia
0,Aguascalientes,101,13.2.1.1,Acacia farnesiana,2,T4_Chis,EXP(-10.01137401+1.97688779*LN(diam)+1.0286075...,1erINF61-85,7.5-132.5,2.5-47.5,...,exp(-10.01137401+1.97688779*ln(diam)+1.0286075...,np.exp(-10.01137401+1.97688779*np.log(diam)+1....,7.5,132.5,2.5,47.5,ok,None,Ecuación para la especie fuera de los límites ...,Publicaciones estatales del primer Inventario ...
1,Aguascalientes,101,13.2.1.1,Acacia pennatula,2,T4_Chis,EXP(-10.01137401+1.97688779*LN(diam)+1.0286075...,1erINF61-85,7.5-132.5,2.5-47.5,...,exp(-10.01137401+1.97688779*ln(diam)+1.0286075...,np.exp(-10.01137401+1.97688779*np.log(diam)+1....,7.5,132.5,2.5,47.5,ok,None,Ecuación para la especie fuera de los límites ...,Publicaciones estatales del primer Inventario ...
2,Aguascalientes,101,13.2.1.1,Arbutus arizonica,2,T15_Chis,EXP(-9.80434696+1.91033696*LN(diam)+1.03262007...,1erINF61-85,7.5-132.5,2.5-47.5,...,exp(-9.80434696+1.91033696*ln(diam)+1.03262007...,np.exp(-9.80434696+1.91033696*np.log(diam)+1.0...,7.5,132.5,2.5,47.5,ok,None,Ecuación para la especie fuera de los límites ...,Publicaciones estatales del primer Inventario ...
3,Aguascalientes,101,13.2.1.1,Arbutus xalapensis,2,T_Dgo_24,"0.0001*Potencia(Diam,1.818)*Potencia(Alt,0.82401)",Siplafor,-,-,...,"0.0001*potencia(diam,1.818)*potencia(alt,0.82401)","0.0001*np.power(diam,1.818)*np.power(alt,0.82401)",NaN,NaN,NaN,NaN,ok,None,Ecuación para la especie fuera de los límites ...,http://fcfposgrado.ujed.mx/spf/inicio/index.php
4,Aguascalientes,101,13.2.1.1,Bursera bipinnata,4,T7_Chis,EXP(-9.86139158+1.93994057*LN(diam)+1.04126898...,1erINF61-85,7.5-132.5,2.5-47.5,...,exp(-9.86139158+1.93994057*ln(diam)+1.04126898...,np.exp(-9.86139158+1.93994057*np.log(diam)+1.0...,7.5,132.5,2.5,47.5,ok,None,Ecuación para el genero fuera de los límites e...,Publicaciones estatales del primer Inventario ...


In [10]:
# Check the overlap of the species lists.

species_in_csv = set(df[COLS.species].dropna().astype(str).unique())
species_in_parquet = set(equations["nombrecientifico_apg_raw"].dropna().astype(str).unique())

missing_species = sorted(species_in_csv - species_in_parquet)

print("Unique species in CSV:", len(species_in_csv))
print("Unique species in parquet:", len(species_in_parquet))
print("Species missing from parquet:", len(missing_species))

missing_species[:25]  # show first 25 missing

Unique species in CSV: 4
Unique species in parquet: 1454
Species missing from parquet: 0


[]

In [11]:
# Estimate tree volume (response variable) for available trees. 

out = estimate_trees(
    df,
    equations=equations,
    estado=ESTADO,
    columns=COLS,
    assignment_level=None,             # None => choose min available (your default rule)
    response_variable=RESPONSE_VARIABLE,
    range_policy=RANGE_POLICY,
    custom_function=None,              # None => use parquet matching
)

out.head(10)

,study_id,site_description,site_condition,Country,Protected area,Location,Site,ManglarIA Project Site ID,IDSitio,Plot ID,...,tree_height_method,estimate_response_variable,match_status,range_status,response_variable,clave_ecuacion,nivel_asignacion,estado_ecuacion_usada,ecuacion_numpy,fuente_referencia
0,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
1,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
2,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
3,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,rod,NaN,dbh_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
4,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
5,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
6,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
7,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,NaN,NaN,height_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN
8,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,rod,0.000088,exact_estado,out_of_range,VRTAcc_m3,T16_Chis,2.0,Campeche,np.exp(-10.12597512+2.04755627*np.log(diam)+0....,Publicaciones estatales del primer Inventario ...
9,Torres_unpub,Petene/degraded,Disturbed,Mexico,Reserva de la Biósfera Ría Lagartos,El Cuyo,Sitio 2,RBRL-EC-09,Cuyo-S01-2,1,...,rod,NaN,dbh_missing,not_evaluated,VRTAcc_m3,NaN,NaN,NaN,NaN,NaN


In [12]:
# Summary
total = len(out)
matched = (out["match_status"] != "no_equation_found").sum()
missing_inputs = out["match_status"].isin(["height_missing", "dbh_missing"]).sum()
no_eq = (out["match_status"] == "no_equation_found").sum()

print(f"Rows: {total}")
print(f"Matched (incl fallback): {matched}")
print(f"Missing inputs: {missing_inputs}")
print(f"No equation found: {no_eq}")

# View key output columns
out[[COLS.species, COLS.dbh_cm, COLS.height_m,
     "estimate_response_variable", "response_variable",
     "match_status", "range_status",
     "estado_ecuacion_usada", "clave_ecuacion", "nivel_asignacion"]].head(20)

Rows: 100
Matched (incl fallback): 100
Missing inputs: 84
No equation found: 0


,Species,DBH (cm),Total Height (m),estimate_response_variable,response_variable,match_status,range_status,estado_ecuacion_usada,clave_ecuacion,nivel_asignacion
0,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
1,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
2,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
3,Avicennia germinans,NaN,1.20,NaN,VRTAcc_m3,dbh_missing,not_evaluated,NaN,NaN,NaN
4,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
5,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
6,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
7,Avicennia germinans,NaN,NaN,NaN,VRTAcc_m3,height_missing,not_evaluated,NaN,NaN,NaN
8,Avicennia germinans,1.114085,1.80,0.000088,VRTAcc_m3,exact_estado,out_of_range,Campeche,T16_Chis,2.0
9,Avicennia germinans,NaN,1.20,NaN,VRTAcc_m3,dbh_missing,not_evaluated,NaN,NaN,NaN
